In [ ]:
#| default_exp analyst_scipy

# Table-attached SciPy analyst (column transforms)

Runs on **in-memory rows** from a Graph-tab table attached in chat.
Generates pandas/numpy/scipy transform code via Lisette and returns
updated rows (new columns) for `loadTableFromChat`.

Serve for the AGE router with:

```bash
pixi run serve-scipy-analyst
```

`POST /analyst/scipy` body: `{ "message": "...", "table": { "rows": [...] } }`.

In [ ]:
#| export
import json
import math
import os
import re
from contextvars import ContextVar
from functools import lru_cache
from typing import Any, TypedDict

import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
from langgraph.graph import END, START, StateGraph
from lisette import Chat, contents

_current_df: ContextVar[pd.DataFrame | None] = ContextVar("scipy_analyst_df", default=None)


def content_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, list):
        return "".join(
            (x.get("text") or "") if isinstance(x, dict) else str(x) for x in content
        )
    return str(content)


def asdict(m: Any) -> dict:
    if isinstance(m, dict):
        return m
    if hasattr(m, "model_dump"):
        return m.model_dump()
    try:
        return dict(m)
    except Exception as ex:  # noqa: BLE001
        raise TypeError(f"cannot convert {type(m)} to dict") from ex

In [ ]:
#| export
class AnalystState(TypedDict, total=False):
    question: str
    columns: str
    result: str
    transform_code: str
    rows: list[dict[str, Any]] | None
    columns_added: list[str]


def get_lisette_llm_config() -> dict[str, str]:
    model = (os.getenv("LISETTE_MODEL") or "").strip()
    key = (os.getenv("LISETTE_API_KEY") or "").strip() or (
        os.getenv("OPENAI_API_KEY") or ""
    ).strip()
    base = (os.getenv("LISETTE_API_BASE") or "").strip()
    if not model:
        return {}
    out: dict[str, str] = {"model": model}
    if key:
        out["api_key"] = key
    if base:
        out["api_base"] = base
    return out


def _active_df() -> pd.DataFrame:
    df = _current_df.get()
    if df is None:
        raise ValueError("No dataframe bound for analyst run")
    return df

In [ ]:
#| export
def analyze_data(state: AnalystState) -> dict:
    df = _active_df()
    cols = list(df.columns.astype(str))
    summary = (
        f"shape={df.shape[0]} rows × {df.shape[1]} cols\n"
        f"dtypes:\n{df.dtypes.to_string()}\n\n"
        f"head:\n{df.head(8).to_string()}"
    )
    result = f"Question: {state.get('question', '')}\n\nData summary:\n{summary}"
    return {"result": result, "columns": ",".join(cols)}


def llm_analyze(state: AnalystState) -> dict:
    prompt = (
        "You are a concise data analyst. Given the question and a small data summary, "
        "describe what column transform would answer the question in 1-3 sentences. "
        "Name the new column(s) you would create. Do not write code.\n\n"
        f"Question: {state.get('question', '')}\n\n"
        f"Summary:\n{state.get('result', '')}"
    )
    cfg = get_lisette_llm_config()
    model = cfg.get("model")
    if not model:
        return {"result": state.get("result", "") + "\n\n(No LISETTE_MODEL; skipping analysis.)"}
    try:
        c = Chat(
            model,
            temp=0,
            api_key=cfg.get("api_key"),
            api_base=cfg.get("api_base"),
        )
        raw_res = c(prompt, max_steps=3, max_tokens=800, return_all=False, stream=False)
        final = contents(raw_res)
        res = content_text(asdict(final).get("content")) if final else ""
    except Exception as e:  # noqa: BLE001
        return {"result": f"{state.get('result', '')}\n\nLLM error: {e}"}
    return {"result": res or state.get("result", "")}


def generate_transform_code(state: AnalystState) -> dict:
    columns = [c for c in (state.get("columns") or "").split(",") if c]
    prompt = f"""You are a pandas + numpy + scipy analyst. Write Python code to transform the user's table.

Question: {state.get('question', '')}

Analysis plan:
{state.get('result', '')}

The dataframe `df` is ALREADY loaded. Columns: {', '.join(columns)}

STRICT RULES:
1. `df` is already available. NEVER recreate or reload data.
2. `pd`, `np`, `scipy`, and `stats` (scipy.stats) are already imported. NEVER import libraries.
3. Mutate `df` in place by adding one or more NEW columns (or overwriting existing ones if asked).
4. Prefer vectorized pandas/numpy/scipy operations over Python loops.
5. Do not drop rows unless the user explicitly asks to filter.
6. NEVER call print(), display(), or write files.
7. Return ONLY Python code — no markdown fences, no commentary.

Write the code now:"""

    cfg = get_lisette_llm_config()
    model = cfg.get("model")
    if not model:
        return {"transform_code": "# No LISETTE_MODEL configured"}
    try:
        c = Chat(
            model,
            temp=0,
            api_key=cfg.get("api_key"),
            api_base=cfg.get("api_base"),
        )
        raw_res = c(prompt, max_steps=3, max_tokens=2000, return_all=False, stream=False)
        final = contents(raw_res)
        text = content_text(asdict(final).get("content")) if final else ""
    except Exception as e:  # noqa: BLE001
        return {"transform_code": f"# LLM error: {e}"}

    code_match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    transform_code = code_match.group(1).strip() if code_match else text.strip()
    return {"transform_code": transform_code}


def _jsonable(obj: Any) -> Any:
    """Convert numpy/pandas scalars into plain JSON-serializable values."""
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if pd.isna(obj):
        return None
    tolist = getattr(obj, "tolist", None)
    if callable(tolist):
        try:
            return _jsonable(tolist())
        except Exception:  # noqa: BLE001
            pass
    item = getattr(obj, "item", None)
    if callable(item):
        try:
            return _jsonable(item())
        except Exception:  # noqa: BLE001
            pass
    if hasattr(obj, "isoformat") and callable(obj.isoformat):
        try:
            return obj.isoformat()
        except Exception:  # noqa: BLE001
            pass
    return str(obj)


def _df_to_rows(df: pd.DataFrame) -> list[dict[str, Any]]:
    records = df.to_dict(orient="records")
    return [_jsonable(r) if isinstance(r, dict) else {} for r in records]


def execute_transform(state: AnalystState) -> dict:
    code = (state.get("transform_code") or "").strip()
    analysis = state.get("result", "") or ""
    if not code or code.startswith("#"):
        return {
            "result": analysis + "\n\nNo transform code generated.",
            "rows": None,
            "columns_added": [],
        }

    before_df = _active_df().copy()
    before_cols = list(before_df.columns.astype(str))
    df = before_df.copy()
    namespace: dict[str, Any] = {
        "df": df,
        "pd": pd,
        "np": np,
        "scipy": scipy,
        "stats": stats,
    }
    try:
        exec(code, namespace)  # noqa: S102
        out_df = namespace.get("df")
        if not isinstance(out_df, pd.DataFrame):
            return {
                "result": analysis + "\n\nTransform error: code did not leave a DataFrame `df`.",
                "rows": None,
                "columns_added": [],
            }
        after_cols = list(out_df.columns.astype(str))
        added = [c for c in after_cols if c not in before_cols]
        modified: list[str] = []
        if not added:
            for c in after_cols:
                if c not in before_cols:
                    continue
                try:
                    if not before_df[c].equals(out_df[c]):
                        modified.append(c)
                except Exception:  # noqa: BLE001
                    modified.append(c)
            if not modified:
                return {
                    "result": analysis + "\n\nTransform error: no columns were added or modified.",
                    "rows": None,
                    "columns_added": [],
                }
            added = modified

        rows = _df_to_rows(out_df)
        if not rows:
            return {
                "result": analysis + "\n\nTransform error: result table is empty.",
                "rows": None,
                "columns_added": added,
            }
        col_note = ", ".join(added)
        return {
            "result": analysis + f"\n\nTransform ready. Columns: {col_note}.",
            "rows": rows,
            "columns_added": added,
        }
    except Exception as e:  # noqa: BLE001
        return {
            "result": analysis + f"\n\nTransform error: {e}",
            "rows": None,
            "columns_added": [],
        }

In [ ]:
#| export
def build_graph():
    graph = StateGraph(AnalystState)
    graph.add_node("analyze", analyze_data)
    graph.add_node("llm_analyze", llm_analyze)
    graph.add_node("generate_transform_code", generate_transform_code)
    graph.add_node("execute_transform", execute_transform)
    graph.add_edge(START, "analyze")
    graph.add_edge("analyze", "llm_analyze")
    graph.add_edge("llm_analyze", "generate_transform_code")
    graph.add_edge("generate_transform_code", "execute_transform")
    graph.add_edge("execute_transform", END)
    return graph.compile()


@lru_cache(maxsize=1)
def get_graph():
    return build_graph()


def run_analyst(question: str, rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Run the SciPy transform analyst on in-memory table rows for the chat UI."""
    if not rows:
        return {
            "status": "error",
            "error": "No table rows to transform.",
            "response": "No table rows to transform.",
            "visualization": "table_scipy",
            "endpoint": "scipy",
            "source": "analyst",
        }
    df = pd.DataFrame(rows)
    token = _current_df.set(df)
    try:
        out = get_graph().invoke(
            {
                "question": (question or "").strip(),
                "columns": ",".join(map(str, df.columns.tolist())),
                "result": "",
                "transform_code": "",
                "rows": None,
                "columns_added": [],
            }
        )
    except Exception as e:  # noqa: BLE001
        return {
            "status": "error",
            "error": str(e),
            "response": f"Analyst failed: {e}",
            "visualization": "table_scipy",
            "endpoint": "scipy",
            "source": "analyst",
        }
    finally:
        _current_df.reset(token)

    out_rows = out.get("rows") if isinstance(out, dict) else None
    response = (out.get("result") if isinstance(out, dict) else None) or ""
    columns_added = (out.get("columns_added") if isinstance(out, dict) else None) or []
    transform_code = (out.get("transform_code") if isinstance(out, dict) else None) or ""
    if isinstance(out_rows, list) and out_rows and isinstance(out_rows[0], dict):
        return {
            "status": "success",
            "response": response,
            "twin_name": "scipy_transform",
            "row_count": len(out_rows),
            "rows": out_rows,
            "columns_added": columns_added,
            "transform_code": transform_code,
            "visualization": "table_scipy",
            "endpoint": "scipy",
            "source": "analyst",
        }
    return {
        "status": "error",
        "error": "No transformed table produced.",
        "response": response or "No transformed table produced.",
        "transform_code": transform_code,
        "visualization": "table_scipy",
        "endpoint": "scipy",
        "source": "analyst",
        "hint": "Try asking to derive or normalize columns using the attached table.",
    }

## HTTP serve

Starlette app for the controller forward target (`hold: table`).

In [ ]:
#| export
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route

ANALYST_PORT = int(os.getenv("ANALYST_PORT", "8013"))
ANALYST_HOST = os.getenv("ANALYST_HOST", "0.0.0.0")


def _rows_from_table(table: Any) -> list[dict[str, Any]]:
    """Extract row dicts from a ``table`` hold payload (list or ``{rows: [...]}``)."""
    if isinstance(table, list):
        return [r for r in table if isinstance(r, dict)]
    if isinstance(table, dict):
        rows = table.get("rows")
        if isinstance(rows, list):
            return [r for r in rows if isinstance(r, dict)]
    return []


async def scipy_endpoint(request: Request) -> JSONResponse:
    """POST /analyst/scipy — run ``run_analyst`` on message + table rows."""
    try:
        body = await request.json()
    except Exception:  # noqa: BLE001
        return JSONResponse({"status": "error", "detail": "Invalid JSON"}, status_code=400)
    if not isinstance(body, dict):
        return JSONResponse({"status": "error", "detail": "JSON object required"}, status_code=400)

    message = body.get("message") or body.get("user_message") or ""
    if not isinstance(message, str) or not message.strip():
        return JSONResponse({"status": "error", "detail": "Missing 'message'"}, status_code=400)

    rows = _rows_from_table(body.get("table"))
    if not rows:
        return JSONResponse(
            {
                "status": "error",
                "error": "Missing or empty 'table' (expected rows).",
                "response": "Missing or empty 'table' (expected rows).",
                "visualization": "table_scipy",
                "endpoint": "scipy",
                "source": "analyst",
            },
            status_code=400,
        )

    result = run_analyst(message.strip(), rows)
    status = 200 if isinstance(result, dict) and result.get("status") == "success" else 502
    return JSONResponse(result if isinstance(result, dict) else {"status": "error"}, status_code=status)


async def health(_: Request) -> JSONResponse:
    """GET /health — liveness for deploy checks."""
    return JSONResponse({"status": "ok", "service": "scipy_analyst"})


app = Starlette(
    routes=[
        Route("/analyst/scipy", scipy_endpoint, methods=["POST"]),
        Route("/health", health, methods=["GET"]),
    ]
)


if __name__ == "__main__":
    import uvicorn

    print(f"Serving scipy analyst on http://{ANALYST_HOST}:{ANALYST_PORT}/analyst/scipy")
    uvicorn.run(app, host=ANALYST_HOST, port=ANALYST_PORT)

In [ ]:
# Demo (not exported): requires LISETTE_* and sample rows
# rows = [{"mwt": 250.0, "logp": 1.2}, {"mwt": 300.0, "logp": 2.1}]
# run_analyst("Add a z-score column of mwt, and a ratio column mwt/logp", rows)